# Smoke test for `new_corr`

This notebook creates small synthetic transcriptomics/proteomics tables and runs:
- `compute_correlation_for_protein`
- `run_correlation_group`

Outputs are written into `./smoke_out/`.


In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "smoke_test"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

from corr_functions import compute_correlation_for_protein, run_correlation_group

np.random.seed(42)

out_dir = Path("smoke_out")
out_dir.mkdir(exist_ok=True)


In [ ]:
# Build synthetic data
n_samples = 20
n_genes = 50
n_prots = 5

samples = [f"S{i:02d}" for i in range(n_samples)]
genes = [f"G{i:03d}" for i in range(n_genes)]
prots = [f"P{i:03d}" for i in range(n_prots)]

# transcriptomics: random noise
X = pd.DataFrame(np.random.normal(size=(n_samples, n_genes)), index=samples, columns=genes)

# make gene G000 strongly related to protein P000
signal = np.random.normal(size=n_samples)
X["G000"] = signal + np.random.normal(scale=0.2, size=n_samples)

# proteomics: random + one correlated
Y = pd.DataFrame(np.random.normal(size=(n_samples, n_prots)), index=samples, columns=prots)
Y["P000"] = signal + np.random.normal(scale=0.2, size=n_samples)

X.index.name = "samples"
Y.index.name = "samples"
X.columns.name = "genes"
Y.columns.name = "proteins"

X.head(), Y.head()


In [ ]:
# Single protein correlation
s = compute_correlation_for_protein(
    transcriptomics=X,
    proteomics=Y,
    protein_name="P000",
    bootstrap=True,
    n_bootstraps=200,
    random_state=123,
)

# show top non-zero correlations
nz = s[s != 0.0].sort_values(key=lambda v: v.abs(), ascending=False)
nz.head(10)


In [ ]:
# Group run (write per-protein CSVs to avoid parquet engine issues)
batch = run_correlation_group(
    proteins=["P000", "P001", "P002"],
    transcriptomics=X,
    proteomics=Y,
    bootstrap=True,
    n_bootstraps=200,
    random_state=42,
    output_dir=out_dir,
    per_protein_format="csv",
)

batch


In [ ]:
# Verify per-protein files exist
per_dir = out_dir / "per_protein"
files = sorted(per_dir.glob("*_corr.csv"))
files


In [ ]:
# Inspect one output file
df_one = pd.read_csv(files[0], index_col=0)
df_one.iloc[:, :10]


In [ ]:
# Decode corr_json for the first protein in the batch
row0 = batch.iloc[0]
corr_dict = json.loads(row0["corr_json"])
list(corr_dict.items())[:10], row0["n_sig"]


## Real data test

In [ ]:
# --- Real-data smoke test (UPDATED: assumes data/ is INSIDE new_corr/) ---
from pathlib import Path
import json

import numpy as np
import pandas as pd
from joblib import load

from corr_functions import compute_correlation_for_protein, run_correlation_group

# Resolve new_corr root whether notebook runs from project root or from ./new_corr
cwd = Path.cwd()

if (cwd / "data").exists():
    # running from inside new_corr (recommended)
    new_corr_root = cwd
elif (cwd / "new_corr" / "data").exists():
    # running from project root
    new_corr_root = cwd / "new_corr"
else:
    # fallback: walk up to find a folder that contains new_corr/data
    new_corr_root = None
    for p in [cwd] + list(cwd.parents):
        if (p / "new_corr" / "data").exists():
            new_corr_root = p / "new_corr"
            break
    if new_corr_root is None:
        raise FileNotFoundError("Could not find 'new_corr/data' from current working directory.")

data_dir = new_corr_root / "data"

trans_path = data_dir / "transcriptomics_filtered.joblib"
prot_path  = data_dir / "proteomics_matched.joblib"
plist_path = data_dir / "protein_list.txt"

assert trans_path.exists(), f"Missing: {trans_path}"
assert prot_path.exists(),  f"Missing: {prot_path}"

print("Loading joblibs...")
X = load(str(trans_path))  # transcriptomics (samples x genes)
Y = load(str(prot_path))   # proteomics (samples x proteins)

# Make sure index/columns naming is consistent (optional but nice)
X.index.name = "samples"
Y.index.name = "samples"
X.columns.name = "genes"
Y.columns.name = "proteins"

print("Shapes:", "X=", X.shape, "Y=", Y.shape)

# Pick a small subset of genes for a quick smoke test (optional, but recommended)
N_GENES = 500
rng = np.random.default_rng(42)
if X.shape[1] > N_GENES:
    gene_subset = rng.choice(X.columns.to_numpy(), size=N_GENES, replace=False)
    X_sm = X.loc[:, gene_subset]
else:
    X_sm = X

# Choose a few proteins (prefer protein_list.txt if present)
def read_protein_list(path: Path):
    return [ln.strip() for ln in path.read_text().splitlines() if ln.strip() and not ln.strip().startswith("#")]

if plist_path.exists():
    candidates = read_protein_list(plist_path)
else:
    candidates = list(Y.columns)

proteins = [p for p in candidates if p in Y.columns][:3]
if len(proteins) == 0:
    raise ValueError("No proteins found to test (protein_list empty or none match Y.columns).")

print("Using proteins:", proteins)
print("Using transcriptomics subset:", X_sm.shape)

# ---- 1) Single-protein function test ----
prot0 = proteins[0]
s = compute_correlation_for_protein(
    transcriptomics=X_sm,
    proteomics=Y,
    protein_name=prot0,
    bootstrap=True,
    n_bootstraps=200,
    random_state=123,
)

nz = s[s != 0.0].sort_values(key=lambda v: v.abs(), ascending=False)
print(f"\nSingle protein: {prot0} | non-zero correlations: {len(nz)}")
display(nz.head(15))

# ---- 2) Group function test (NO saving; output_dir=None) ----
batch = run_correlation_group(
    proteins=proteins,
    transcriptomics=X_sm,
    proteomics=Y,
    bootstrap=True,
    n_bootstraps=200,
    random_state=42,
    output_dir=None,  # IMPORTANT: no files written
)

print("\nBatch summary:")
display(batch)

# Inspect decoded corr_json for the first protein
corr_dict0 = json.loads(batch.loc[0, "corr_json"])
print(f"\nDecoded corr_json for {batch.loc[0, 'protein']} (showing up to 10 items):")
display(list(corr_dict0.items())[:10])

# ---- (Optional) If we want to write outputs like the pipeline does ----
# out_dir = data_dir / "results"
# out_dir.mkdir(parents=True, exist_ok=True)
# batch_saved = run_correlation_group(
#     proteins=proteins,
#     transcriptomics=X_sm,
#     proteomics=Y,
#     bootstrap=True,
#     n_bootstraps=200,
#     random_state=42,
#     output_dir=out_dir,
#     per_protein_format="parquet",
# )
# display(batch_saved)
# print("Wrote per-protein files under:", out_dir / "per_protein")
